# HTML, CSS et l'inspecteur

## Etape 1
Site internet : books.toscrape.com

Catégorie choisie : Food and Drink

Nombre de livres : 30 livres

Nombre de pages : 2
 
## Etape 2
Saisie dans Console : (le sélecteur = article.product_pod)

`document.querySelectorAll('article.product_pod').length`

Résultat : 20
 
## Etape 3
| Donnée             | Sélecteur              |	Contenu ou attribut ? |
|--------------------|------------------------|------------------------|
|titre               | `article h3 a`                   | Attribut title |
|prix                | `article p.price_color`          |	Contenu |
|dispo               | `article p.instock.availability` | Contenu |
|note                | `article p.star-rating`          | Attribut class (1 à 5)|
|lien vers la fiche  | `article h3 a`                   | Attribut href |
 
 ## Etape 4
`document.querySelectorAll('article p.price_color').length`

Résultat : 20

`document.querySelectorAll('article h3 a').length`

Résultat : 20
 
## Etape 5
Selecteur = `li.next a`

Attribut = href

Valeur = "page-2.html" (à coller à la fin de l'adresse de la page courante)
 
## Etape 6
Affichage du prix dans l'inspecteur : `#content_inner > article > div.row > div.col-sm-6.product_main > p.price_color`

Affichage du prix dans le code source :  `<p class="price_color">£37.34</p>`

Le prix est présent dans les deux. Donc le site est statique rien n'est ajouté après coup. Un script avec requests trouvera bien l'information.
 
## Etape 7
Informations présentes sur la page du livre mais absentes de la page de la liste des livres : 
-  Quantité du stock dispo = `tbody tr td` (th = "Availability")
-  Description du produit = `#product_description+p` (test ok avec la commande `document.querySelectorAll('#product_description+p')[0].textContent`)

In [ ]:
# Etape 1 
import requests

URL = "https://books.toscrape.com/catalogue/category/books/food-and-drink_33/index.html"

ENTETES = {"User-Agent": "bootcamp-data/1.0 (fontes.benjamin@gmail.com)"}

r = requests.get(URL, headers=ENTETES, timeout=10)

r.encoding = "utf-8"

print("status   :", r.status_code)
print("longueur :", len(r.text), "caracteres")
print("debut    :", r.text[:60])

status   : 200
longueur : 54104 caracteres
debut    : 

<!DOCTYPE html>
<!--[if lt IE 7]>      <html lang="en-us" 


In [31]:
# Test pour vérifier si la page visée est la bonne

print("le prix est present :", "£30.52" in r.text)
print("le titre est present:", "Foolproof" in r.text)

le prix est present : True
le titre est present: True


In [ ]:

print("fiches trouvees :", len(fiches))

In [33]:
from urllib.parse import urljoin
import pandas as pd

# Traduction des notes en chiffres
NOTES = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}

lignes = []

for fiche in fiches:
   prix = fiche.select_one("p.price_color").get_text(strip=True)        # Contenu d'une balise 
   dispo = fiche.select_one("p.instock").get_text(strip=True)           # Contenu d'une balise 
   lien_h3 = fiche.select_one("h3 a")                                   # Attribut
   titre = lien_h3["title"]                                             # Attribut
   href = lien_h3["href"]                                               # Attribut
   classes = fiche.select_one("p.star-rating")["class"]                 # Lire une classe
   note = NOTES[classes[1]]
   lignes.append({
        "titre": titre, 
        "prix": prix, 
        "dispo": dispo, 
        "note": note, 
        "url": urljoin(URL, href)
    }
    )

df = pd.DataFrame(lignes)
df

,titre,prix,dispo,note,url
0,Foolproof Preserving: A Guide to Small Batch J...,£30.52,In stock,3,https://books.toscrape.com/catalogue/foolproof...
1,The Pioneer Woman Cooks: Dinnertime: Comfort C...,£56.41,In stock,1,https://books.toscrape.com/catalogue/the-pione...
2,My Paris Kitchen: Recipes and Stories,£33.37,In stock,2,https://books.toscrape.com/catalogue/my-paris-...
3,Mama Tried: Traditional Italian Cooking for th...,£14.02,In stock,4,https://books.toscrape.com/catalogue/mama-trie...
4,"Layered: Baking, Building, and Styling Spectac...",£40.11,In stock,1,https://books.toscrape.com/catalogue/layered-b...
5,The Nerdy Nummies Cookbook: Sweet Treats for t...,£37.34,In stock,5,https://books.toscrape.com/catalogue/the-nerdy...
6,The Love and Lemons Cookbook: An Apple-to-Zucc...,£37.60,In stock,2,https://books.toscrape.com/catalogue/the-love-...
7,The Cookies & Cups Cookbook: 125+ sweet & savo...,£41.25,In stock,1,https://books.toscrape.com/catalogue/the-cooki...
8,Deliciously Ella Every Day: Quick and Easy Rec...,£42.16,In stock,3,https://books.toscrape.com/catalogue/delicious...
9,The Help Yourself Cookbook for Kids: 60 Easy P...,£28.77,In stock,3,https://books.toscrape.com/catalogue/the-help-...


In [34]:
# Vérification des dimensions du dataframe (lignes, colonnes)
df.shape

(20, 5)

In [35]:
# Véirification aucune donnée tronquée (somme des "..." dans les titres)
df["titre"].str.contains("...", regex=False).sum()

np.int64(0)

In [36]:
# Vérification des notes
df["note"].value_counts()

note
1    6
3    4
2    4
5    4
4    2
Name: count, dtype: int64

In [ ]:
import time

from bs4 import BeautifulSoup

url = "https://books.toscrape.com/catalogue/category/books/food-and-drink_33/index.html"
lignes = []
page = 0
pages_max = 5


while url is not None and page < pages_max:
    # Télécharger et parser la page
    r = requests.get(url, headers=ENTETES, timeout=10)
    r.encoding = "utf-8"
    soup = BeautifulSoup(r.text, "html.parser")
    fiches = soup.select("article.product_pod")

    # Extraire les livres
    for fiche in fiches:
        prix = fiche.select_one("p.price_color").get_text(strip=True)        # Contenu d'une balise 
        dispo = fiche.select_one("p.instock").get_text(strip=True)           # Contenu d'une balise 
        lien_h3 = fiche.select_one("h3 a")                                   # Attribut
        titre = lien_h3["title"]                                             # Attribut
        href = lien_h3["href"]                                               # Attribut
        classes = fiche.select_one("p.star-rating")["class"]                 # Lire une classe
        note = NOTES[classes[1]]
        lignes.append({
            "titre": titre, 
            "prix": prix, 
            "dispo": dispo, 
            "note": note, 
            "url": urljoin(url, href)
        }
        )

    # Préparer la page suivante 
    page_suivante = soup.select_one("li.next a")
    if page_suivante is None:
        url = None
    else:
        url = urljoin(url, page_suivante["href"])


    # Compteur et pause
    page += 1
    time.sleep(1)

print(f" Nombre total de livres : {len(lignes)}")

 Nombre de livres : 30


In [ ]:
# url du 1er livre
url_livre = lignes[0]["url"]

# Dl et parser la page
r = requests.get(url_livre, headers=ENTETES, timeout=10)
r.encoding = "utf-8"
soup_livre = BeautifulSoup(r.text, "html.parser")

# Extraction des données
cellules = soup_livre.select("table.table-striped tr td")
categorie = soup_livre.select("ul.breadcrumb li a")[2].get_text(strip=True)
upc = cellules[0].get_text(strip=True)
stock = cellules[5].get_text(strip=True)

print(categorie, upc, stock)
    

Food and Drink 5674a18a29a43ced In stock (19 available)


In [49]:
def lire_detail(url):
    r = requests.get(url, headers=ENTETES, timeout=10)
    r.encoding = "utf-8"
    if r.status_code != 200:
        return {"categorie": None, "stock": None}

    soup = BeautifulSoup(r.text, "html.parser")

    # le tableau : une entree par ligne, le th en cle et le td en valeur
    infos = {}
    for ligne in soup.select("table.table-striped tr"):
        cle = ligne.select_one("th")
        val = ligne.select_one("td")
        if cle and val:
            infos[cle.get_text(strip=True)] = val.get_text(strip=True)

    # la categorie : troisieme element du fil d'Ariane
    miettes = soup.select("ul.breadcrumb li a")
    categorie = miettes[-1].get_text(strip=True) if miettes else None

    return {
        "categorie": categorie,
        "stock": infos.get("Availability"),
        "upc": infos.get("UPC"),
    }


url_test = "https://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html"
print(lire_detail(lignes[0]["url"]))

for i, livre in enumerate(lignes, start=1):
    detail = lire_detail(livre["url"])
    livre.update(detail)
    print(f"livre {i}/{len(lignes)} : {detail['categorie']}")
    time.sleep(1)

{'categorie': 'Food and Drink', 'stock': 'In stock (19 available)', 'upc': '5674a18a29a43ced'}
livre 1/30 : Food and Drink
livre 2/30 : Food and Drink
livre 3/30 : Food and Drink
livre 4/30 : Food and Drink
livre 5/30 : Food and Drink
livre 6/30 : Food and Drink
livre 7/30 : Food and Drink
livre 8/30 : Food and Drink
livre 9/30 : Food and Drink
livre 10/30 : Food and Drink
livre 11/30 : Food and Drink
livre 12/30 : Food and Drink
livre 13/30 : Food and Drink
livre 14/30 : Food and Drink
livre 15/30 : Food and Drink
livre 16/30 : Food and Drink
livre 17/30 : Food and Drink
livre 18/30 : Food and Drink
livre 19/30 : Food and Drink
livre 20/30 : Food and Drink
livre 21/30 : Food and Drink
livre 22/30 : Food and Drink
livre 23/30 : Food and Drink
livre 24/30 : Food and Drink
livre 25/30 : Food and Drink
livre 26/30 : Food and Drink
livre 27/30 : Food and Drink
livre 28/30 : Food and Drink
livre 29/30 : Food and Drink
livre 30/30 : Food and Drink


In [51]:
df = pd.DataFrame(lignes)
df

,titre,prix,dispo,note,url,categorie,stock,upc
0,Foolproof Preserving: A Guide to Small Batch J...,£30.52,In stock,3,https://books.toscrape.com/catalogue/foolproof...,Food and Drink,In stock (19 available),5674a18a29a43ced
1,The Pioneer Woman Cooks: Dinnertime: Comfort C...,£56.41,In stock,1,https://books.toscrape.com/catalogue/the-pione...,Food and Drink,In stock (16 available),142d767a56a9254d
2,My Paris Kitchen: Recipes and Stories,£33.37,In stock,2,https://books.toscrape.com/catalogue/my-paris-...,Food and Drink,In stock (16 available),784bc2279baa878d
3,Mama Tried: Traditional Italian Cooking for th...,£14.02,In stock,4,https://books.toscrape.com/catalogue/mama-trie...,Food and Drink,In stock (16 available),72f9d5be3472d34e
4,"Layered: Baking, Building, and Styling Spectac...",£40.11,In stock,1,https://books.toscrape.com/catalogue/layered-b...,Food and Drink,In stock (16 available),7b75cc7f223791f4
5,The Nerdy Nummies Cookbook: Sweet Treats for t...,£37.34,In stock,5,https://books.toscrape.com/catalogue/the-nerdy...,Food and Drink,In stock (15 available),e4c7f3487d17fd42
6,The Love and Lemons Cookbook: An Apple-to-Zucc...,£37.60,In stock,2,https://books.toscrape.com/catalogue/the-love-...,Food and Drink,In stock (15 available),e28e330cbdf01126
7,The Cookies & Cups Cookbook: 125+ sweet & savo...,£41.25,In stock,1,https://books.toscrape.com/catalogue/the-cooki...,Food and Drink,In stock (15 available),96a38e7c813d8f1d
8,Deliciously Ella Every Day: Quick and Easy Rec...,£42.16,In stock,3,https://books.toscrape.com/catalogue/delicious...,Food and Drink,In stock (15 available),46c1530d7546ea6d
9,The Help Yourself Cookbook for Kids: 60 Easy P...,£28.77,In stock,3,https://books.toscrape.com/catalogue/the-help-...,Food and Drink,In stock (14 available),ddc8340dabc8641b
